# Lab 0 — Build & Push GR00T Training Container to ECR

**Run this notebook once before `Lab1_Training_Inference.ipynb`.**  
It builds a Docker image with the Isaac-GR00T SDK pre-installed and pushes it to ECR.
After a successful build, the image is reusable for all training jobs.

**Read `BACKGROUND.md` first** for context on what GR00T is and how this lab works.

---

## What this notebook does
1. Verifies all AWS prerequisites (CodeBuild, ECR, S3 access)
2. Creates the ECR repository if it doesn't exist
3. Packages the container files (`Dockerfile`, training entrypoint, modality config) into a zip
4. Uploads the zip to S3 as the CodeBuild source
5. Creates a CodeBuild project and triggers a build (~20-30 min)
6. Monitors the build until completion

## Why CodeBuild instead of local Docker?
- The NVIDIA CUDA base image is 5+ GB — slow to pull on a laptop
- SageMaker JupyterLab blocks outbound network inside `docker build`
- CodeBuild runs on a large cloud instance with fast network, privileged Docker

## IAM Requirements (must be done before running cells)

Add these to your **SageMaker execution role** in the AWS IAM Console:

### Attached policies
| Policy | Why needed |
|--------|-----------|
| `AWSCodeBuildDeveloperAccess` | Create/start CodeBuild projects |
| `AmazonEC2ContainerRegistryFullAccess` | Push image to ECR |
| `AmazonS3FullAccess` | Upload build context zip |

### Inline policy (add manually)
```json
{
  "Version": "2012-10-17",
  "Statement": [{"Effect": "Allow", "Action": "iam:PassRole", "Resource": "*"}]
}
```

### Trust relationship (edit in IAM Console)
The role must trust both SageMaker and CodeBuild:
```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": {"Service": ["sagemaker.amazonaws.com", "codebuild.amazonaws.com"]},
    "Action": "sts:AssumeRole"
  }]
}
```

> **Admin access required** to make these IAM changes.


## 0 — Setup

In [1]:
import boto3, io, json, time, zipfile
from pathlib import Path

# ── Resolve paths ─────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "aws-physical-ai-toolchain" and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

LAB1_DIR       = REPO_ROOT / "notebooks" / "lab1"
CONTAINER_DIR  = LAB1_DIR / "container"   # Dockerfile and entrypoint scripts

# ── AWS config ────────────────────────────────────────────────────────────
REGION   = boto3.session.Session().region_name or "us-west-2"
ACCOUNT  = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET   = f"sagemaker-{REGION}-{ACCOUNT}"
ROLE_ARN = f"arn:aws:iam::{ACCOUNT}:role/service-role/YOUR_SAGEMAKER_EXECUTION_ROLE"
ECR_REPO = "physical-ai/groot-training"
ECR_URI  = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}"

print(f"Region:        {REGION}")
print(f"Account:       {ACCOUNT}")
print(f"Bucket:        {BUCKET}")
print(f"Role ARN:      {ROLE_ARN}")
print(f"ECR URI:       {ECR_URI}")
print(f"Container dir: {CONTAINER_DIR}")


Region:        us-west-2
Account:       YOUR_ACCOUNT_ID
Bucket:        sagemaker-us-west-2-YOUR_ACCOUNT_ID
Role ARN:      arn:aws:iam::YOUR_ACCOUNT_ID:role/service-role/YOUR_SAGEMAKER_EXECUTION_ROLE
ECR URI:       YOUR_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/physical-ai/groot-training
Container dir: /home/sagemaker-user/aws-physical-ai-toolchain/lab1/container


## 1 — Verify Prerequisites

In [2]:
errors = []
ecr = boto3.client("ecr", region_name=REGION)
cb  = boto3.client("codebuild", region_name=REGION)
s3  = boto3.client("s3", region_name=REGION)

# CodeBuild access
try:
    cb.list_projects()
    print("✅ CodeBuild access OK")
except Exception as e:
    errors.append(str(e))
    print(f"❌ CodeBuild access MISSING\n   Attach AWSCodeBuildDeveloperAccess to the role\n   Error: {e}")

# ECR access
try:
    ecr.describe_repositories(repositoryNames=[ECR_REPO])
    print(f"✅ ECR repo '{ECR_REPO}' exists")
except ecr.exceptions.RepositoryNotFoundException:
    print(f"ℹ️  ECR repo '{ECR_REPO}' not found — will be created in next cell")
except Exception as e:
    errors.append(str(e))
    print(f"❌ ECR access MISSING: {e}")

# S3 access
try:
    s3.head_bucket(Bucket=BUCKET)
    print(f"✅ S3 bucket '{BUCKET}' accessible")
except Exception as e:
    errors.append(str(e))
    print(f"❌ S3 access issue: {e}")

# Container files
required = ["Dockerfile", "train_entrypoint.py", "ur3_modality_config.py", "video_utils_patched.py"]
for fname in required:
    fpath = CONTAINER_DIR / fname
    if fpath.exists():
        print(f"✅ {fname:35s} ({fpath.stat().st_size // 1024} KB)")
    else:
        errors.append(f"Missing: {fpath}")
        print(f"❌ MISSING: {fpath}")

print()
if errors:
    print("⚠️  Fix the issues above before proceeding.")
else:
    print("All checks passed. Continue to next cell.")


✅ CodeBuild access OK
✅ ECR repo 'physical-ai/groot-training' exists
✅ S3 bucket 'sagemaker-us-west-2-YOUR_ACCOUNT_ID' accessible
✅ Dockerfile                          (5 KB)
✅ train_entrypoint.py                 (17 KB)
✅ ur3_modality_config.py              (2 KB)
✅ video_utils_patched.py              (5 KB)

All checks passed. Continue to next cell.


## 2 — Create ECR Repository (if needed)

In [3]:
try:
    repo = ecr.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
    print(f"✅ Already exists: {repo['repositoryUri']}")
except ecr.exceptions.RepositoryNotFoundException:
    repo = ecr.create_repository(repositoryName=ECR_REPO)["repository"]
    print(f"✅ Created: {repo['repositoryUri']}")


✅ Already exists: YOUR_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/physical-ai/groot-training


## 3 — Package Container Files & Upload to S3

The build context (Dockerfile + supporting scripts) is zipped and uploaded to S3.
CodeBuild downloads this zip as its source — no GitHub connection required.

**Container files in `notebooks/lab1/container/`:**

| File | Purpose |
|------|---------|
| `Dockerfile` | Defines the image: CUDA 12.4 + Python 3.11 + PyTorch 2.5 + Isaac-GR00T SDK |
| `train_entrypoint.py` | Runs inside the container when SageMaker starts a training job |
| `ur3_modality_config.py` | Registers the UR3 robot's sensor/action layout with the GR00T SDK |
| `video_utils_patched.py` | PyAV fallback for video decoding (torchcodec unavailable in some envs) |

The `Dockerfile` clones `https://github.com/NVIDIA/Isaac-GR00T` at build time and
installs the SDK with pinned dependencies. **No git clone is needed on your machine.**


In [4]:
S3_KEY = "codebuild/groot-training-context.zip"

# Read the correct buildspec from file (handles DLC ECR auth)
# Adjust 'cd containers/groot-training' → removed since files are at zip root
buildspec_path = CONTAINER_DIR / 'buildspec.yml'
if buildspec_path.exists():
    BUILDSPEC = buildspec_path.read_text()
    # Remove 'cd containers/groot-training' since all files are at zip root
    BUILDSPEC = BUILDSPEC.replace('      - cd containers/groot-training\n', '')
    print('✅ Using buildspec.yml from container dir')
else:
    raise FileNotFoundError(f'buildspec.yml not found at {buildspec_path}')

buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('buildspec.yml', BUILDSPEC)
    for fname in ['Dockerfile', 'train_entrypoint.py',
                  'ur3_modality_config.py', 'video_utils_patched.py']:
        fpath = CONTAINER_DIR / fname
        if fpath.exists():
            zf.write(fpath, fname)
            print(f'  Added: {fname} ({fpath.stat().st_size // 1024} KB)')
        else:
            print(f'  WARNING: {fname} not found')
buf.seek(0)

s3.upload_fileobj(buf, BUCKET, S3_KEY)
print(f'\n✅ Build context uploaded: s3://{BUCKET}/{S3_KEY}')


✅ Using buildspec.yml from container dir
  Added: Dockerfile (5 KB)
  Added: train_entrypoint.py (17 KB)
  Added: ur3_modality_config.py (2 KB)
  Added: video_utils_patched.py (5 KB)

✅ Build context uploaded: s3://sagemaker-us-west-2-YOUR_ACCOUNT_ID/codebuild/groot-training-context.zip


## 4 — Create CodeBuild Project (idempotent)

In [5]:
PROJECT = "physical-ai-groot-training-build"

project_cfg = dict(
    source={"type": "S3", "location": f"{BUCKET}/{S3_KEY}", "buildspec": "buildspec.yml"},
    artifacts={"type": "NO_ARTIFACTS"},
    environment={
        "type": "LINUX_CONTAINER",
        "image": "aws/codebuild/standard:7.0",
        "computeType": "BUILD_GENERAL1_XLARGE",   # 72 GB RAM, fast network
        "privilegedMode": True,                    # required for docker build
        "environmentVariables": [
            {"name": "ECR_REPO_URI",       "value": ECR_URI},
            {"name": "AWS_DEFAULT_REGION", "value": REGION},
        ],
    },
    serviceRole=ROLE_ARN,
    timeoutInMinutes=60,
)

try:
    cb.create_project(name=PROJECT, **project_cfg)
    print(f"✅ Project created: {PROJECT}")
except cb.exceptions.ResourceAlreadyExistsException:
    cb.update_project(name=PROJECT, **{k:v for k,v in project_cfg.items() if k != "timeoutInMinutes"})
    print(f"✅ Project updated: {PROJECT}")


✅ Project updated: physical-ai-groot-training-build


## 5 — Start Build

In [6]:
build    = cb.start_build(projectName=PROJECT)
BUILD_ID = build["build"]["id"]

print(f"✅ Build started!")
print(f"   ID:      {BUILD_ID}")
print(f"   Status:  {build['build']['buildStatus']}")
print()
print("Build stages (~20-30 min total):")
print("  1. Pull nvidia/cuda:12.4.1-devel-ubuntu22.04 base (~5 GB)")
print("  2. Install Python 3.11 + PyTorch 2.5.1 (CUDA 12.4 wheels)")
print("  3. Clone Isaac-GR00T SDK from github.com/NVIDIA/Isaac-GR00T")
print("  4. Install SDK dependencies with pinned versions")
print("  5. Copy entrypoint + modality config + video utils into image")
print("  6. Push final image (~7.8 GB) to ECR")
print()
print("Monitor in AWS Console:")
print(f"  https://{REGION}.console.aws.amazon.com/codesuite/codebuild/projects/{PROJECT}/history?region={REGION}")
print()
print("Or run the next cell to poll status here.")


✅ Build started!
   ID:      physical-ai-groot-training-build:583a140e-5663-4798-b633-432cce2343e5
   Status:  IN_PROGRESS

Build stages (~20-30 min total):
  1. Pull nvidia/cuda:12.4.1-devel-ubuntu22.04 base (~5 GB)
  2. Install Python 3.11 + PyTorch 2.5.1 (CUDA 12.4 wheels)
  3. Clone Isaac-GR00T SDK from github.com/NVIDIA/Isaac-GR00T
  4. Install SDK dependencies with pinned versions
  5. Copy entrypoint + modality config + video utils into image
  6. Push final image (~7.8 GB) to ECR

Monitor in AWS Console:
  https://us-west-2.console.aws.amazon.com/codesuite/codebuild/projects/physical-ai-groot-training-build/history?region=us-west-2

Or run the next cell to poll status here.


## 6 — Poll Build Status

Re-run this cell every 2-3 minutes until `Status: SUCCEEDED`.

In [10]:
b       = cb.batch_get_builds(ids=[BUILD_ID])["builds"][0]
status  = b["buildStatus"]
phase   = b.get("currentPhase", "")
elapsed = int((time.time() - b["startTime"].timestamp()) / 60)

print(f"Status:  {status}")
print(f"Phase:   {phase}")
print(f"Elapsed: {elapsed} min")

if status == "SUCCEEDED":
    imgs = ecr.describe_images(repositoryName=ECR_REPO,
                               imageIds=[{"imageTag": "latest"}])["imageDetails"]
    img  = imgs[0]
    print(f"\n✅ Build SUCCEEDED!")
    print(f"   Image size:   {img['imageSizeInBytes'] / 1e9:.1f} GB")
    print(f"   Pushed at:    {img['imagePushedAt'].strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"   ECR URI:      {ECR_URI}:latest")
    print()
    print("You can now run Lab1_Training_Inference.ipynb")

elif status == "FAILED":
    print("\n❌ Build FAILED. Check phases:")
    for ph in b.get("phases", []):
        if ph.get("phaseStatus") == "FAILED":
            print(f"  Phase: {ph['phaseType']}")
            for ctx in ph.get("contexts", []):
                print(f"  Error: {ctx.get('message','')}")
    print(f"\nFull logs:")
    print(f"  https://{REGION}.console.aws.amazon.com/codesuite/codebuild/projects/{PROJECT}/history?region={REGION}")
else:
    print("\n(Re-run this cell in ~2-3 min to check again)")


Status:  SUCCEEDED
Phase:   COMPLETED
Elapsed: 17 min

✅ Build SUCCEEDED!
   Image size:   16.6 GB
   Pushed at:    2026-06-30 23:59 UTC
   ECR URI:      YOUR_ACCOUNT_ID.dkr.ecr.us-west-2.amazonaws.com/physical-ai/groot-training:latest

You can now run Lab1_Training_Inference.ipynb


## 7 — Rebuild After Changes (optional)

Run this if you modify `Dockerfile` or `train_entrypoint.py`.

In [ ]:
# Re-upload build context with latest files, then start a new build
buildspec_path = CONTAINER_DIR / 'buildspec.yml'
BUILDSPEC_CONTENT = buildspec_path.read_text().replace(
    '      - cd containers/groot-training\n', '')

buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('buildspec.yml', BUILDSPEC_CONTENT)
    for fname in ['Dockerfile', 'train_entrypoint.py',
                  'ur3_modality_config.py', 'video_utils_patched.py']:
        fpath = CONTAINER_DIR / fname
        if fpath.exists(): zf.write(fpath, fname)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, S3_KEY)

build    = cb.start_build(projectName=PROJECT)
BUILD_ID = build['build']['id']
print(f'✅ Rebuild started: {BUILD_ID}')
print('Re-run the poll cell to monitor.')
